# 🏥 Patient Readmission Prediction
## Step 2: Data Preprocessing

Steps covered:
- Handle missing values
- Encode categorical variables
- Feature engineering
- Handle class imbalance with SMOTE
- Save clean dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import pickle
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded!')

## 2.1 Load & Initial Clean

In [ ]:
df = pd.read_csv('../data/diabetic_data.csv')
print(f'Original shape: {df.shape}')

# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)

# Drop columns with >40% missing or not useful
cols_to_drop = ['encounter_id', 'patient_nbr', 'weight', 
                'payer_code', 'medical_specialty',
                'diag_1', 'diag_2', 'diag_3']  # raw ICD codes — too many categories
df.drop(columns=cols_to_drop, inplace=True)

# Drop rows with remaining NaN (small %)
df.dropna(inplace=True)
print(f'After cleaning: {df.shape}')

## 2.2 Target Variable

In [ ]:
# Binary: readmitted within 30 days = 1, else = 0
df['readmitted'] = (df['readmitted'] == '<30').astype(int)
print(f'Target distribution:\n{df["readmitted"].value_counts()}')

## 2.3 Feature Engineering

In [ ]:
# Age: convert range to midpoint number
age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75,
    '[80-90)': 85, '[90-100)': 95
}
df['age'] = df['age'].map(age_map)

# New features
df['total_visits'] = df['number_outpatient'] + df['number_emergency'] + df['number_inpatient']
df['medication_per_day'] = df['num_medications'] / df['time_in_hospital']

print('Feature engineering done!')
print(f'New features added: total_visits, medication_per_day')

## 2.4 Encode Categorical Variables

In [ ]:
# Binary categorical columns
binary_cols = ['change', 'diabetesMed']
for col in binary_cols:
    df[col] = (df[col] == 'Ch').astype(int) if col == 'change' else (df[col] == 'Yes').astype(int)

# Gender
df['gender'] = (df['gender'] == 'Male').astype(int)

# Medication columns (many: Up/Down/Steady/No)
med_cols = ['metformin','repaglinide','nateglinide','chlorpropamide','glimepiride',
            'acetohexamide','glipizide','glyburide','tolbutamide','pioglitazone',
            'rosiglitazone','acarbose','miglitol','troglitazone','tolazamide',
            'examide','citoglipton','insulin','glyburide-metformin','glipizide-metformin',
            'glimepiride-pioglitazone','metformin-rosiglitazone','metformin-pioglitazone']

dose_map = {'No': 0, 'Steady': 1, 'Up': 2, 'Down': 2}
for col in med_cols:
    if col in df.columns:
        df[col] = df[col].map(dose_map).fillna(0).astype(int)

# A1Cresult and max_glu_serum
a1c_map = {'None': 0, 'Norm': 1, '>7': 2, '>8': 3}
glu_map = {'None': 0, 'Norm': 1, '>200': 2, '>300': 3}
df['A1Cresult'] = df['A1Cresult'].map(a1c_map).fillna(0)
df['max_glu_serum'] = df['max_glu_serum'].map(glu_map).fillna(0)

# Race — label encode
le = LabelEncoder()
df['race'] = le.fit_transform(df['race'].astype(str))

print(f'Encoding complete. Final shape: {df.shape}')
print(f'Remaining dtypes: {df.dtypes.value_counts().to_dict()}')

## 2.5 Train-Test Split + SMOTE

In [ ]:
X = df.drop('readmitted', axis=1)
y = df['readmitted']

# Save feature names for later use
feature_names = X.columns.tolist()
with open('../data/feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {X_train.shape}')
print(f'Test size:  {X_test.shape}')
print(f'\nBefore SMOTE — Class distribution:\n{y_train.value_counts()}')

# SMOTE for class imbalance
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'\nAfter SMOTE — Class distribution:\n{pd.Series(y_train_res).value_counts()}')

## 2.6 Save Processed Data

In [ ]:
import pickle

data_bundle = {
    'X_train': X_train_res,
    'X_test': X_test,
    'y_train': y_train_res,
    'y_test': y_test,
    'feature_names': feature_names
}

with open('../data/processed_data.pkl', 'wb') as f:
    pickle.dump(data_bundle, f)

print('✅ Processed data saved to data/processed_data.pkl')
print('\nProceed to 03_model.ipynb')